In [ ]:
import os
import jiwer
import numpy as np
from faster_whisper import WhisperModel

In [ ]:
print("🚀 Loading Faster-Whisper (Base)...")
model = WhisperModel("base.en", device="cpu", compute_type="int8")
results = []

In [ ]:
def validate_tts_with_asr(wav_directory: str, reference_texts: dict):
    """
    Transcribes a folder of generated TTS wav files and calculates corpus-level macro-averaged WER, WRR, and CER.
    
    :param wav_directory: Path to the folder containing your 50 generated .wav files.
    :param reference_texts: A dictionary mapping the filename to the original clean text.
                            e.g., {"test_matcha.wav": "I wanted to find the word."}
    """
    results = []
    all_clean_refs = []
    all_clean_hyps = []
    
    # 1. Text standardization transformation pipeline
    transformation = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
    ])

    for filename in os.listdir(wav_directory):
        if not filename.endswith(".wav"):
            continue
            
        filepath = os.path.join(wav_directory, filename)
        
        if filename not in reference_texts:
            print(f"⚠️ Skipping {filename}: No reference text provided.")
            continue
            
        reference = reference_texts[filename]
        
        # Transcribe the audio
        segments, info = model.transcribe(filepath, beam_size=5)
        hypothesis = " ".join([segment.text for segment in segments]).strip()
        
        # Standardize text for fair comparison
        clean_ref = transformation(reference)
        clean_hyp = transformation(hypothesis)
        
        # Collect for corpus-level macro calculation
        all_clean_refs.append(clean_ref)
        all_clean_hyps.append(clean_hyp)
        
        # Handle empty strings to avoid division by zero errors
        try:
            wer = jiwer.wer(clean_ref, clean_hyp) if clean_ref else 0.0
            cer = jiwer.cer(clean_ref, clean_hyp) if clean_ref else 0.0
        except ValueError:
            wer, cer = 1.0, 1.0
            
        wrr = max(0.0, 1.0 - wer)
        
        print(f"\n🎧 File: {filename}")
        print(f"   Truth: {clean_ref}")
        print(f"   ASR:   {clean_hyp}")
        print(f"   📊 Per-file WER: {wer:.1%} | WRR: {wrr:.1%} | CER: {cer:.1%}")
        
        results.append({
            "filename": filename,
            "wer": wer,
            "wrr": wrr,
            "cer": cer,
            "ref": clean_ref,
            "hyp": clean_hyp
        })

    # 2. Compute True Corpus-Level Macro Averages (Aggregated across all sentences)
    corpus_wer = jiwer.wer(all_clean_refs, all_clean_hyps)
    corpus_cer = jiwer.cer(all_clean_refs, all_clean_hyps)
    corpus_wrr = max(0.0, 1.0 - corpus_wer)

    # Individual sentence arrays for your downstream plotting lists
    wer_list = [r["wer"] for r in results]
    cer_list = [r["cer"] for r in results]

    print("\n" + "="*40)
    print("🏆 FINAL CORPUS-LEVEL BENCHMARK 🏆")
    print(f"Overall Corpus WER: {corpus_wer:.1%}")
    print(f"Overall Corpus CER: {corpus_cer:.1%}")
    print(f"Overall Corpus WRR: {corpus_wrr:.1%}")
    print("="*40)

    return {
        "average_wer": float(corpus_wer),
        "average_cer": float(corpus_cer),
        "raw_wer_list": wer_list,
        "raw_cer_list": cer_list,
        "file_details": results
    }

In [ ]:
def load_ground_truth_map(txt_filepath):
    """
    Reads a text file line by line and maps it to script_01.wav, script_02.wav, etc.
    """
    ground_truth = {}
    
    with open(txt_filepath, 'r', encoding='utf-8') as file:
        lines = [line.strip() for line in file.readlines() if line.strip()]
        
    for index, text in enumerate(lines):
        filename = f"script_{index + 1:02d}.wav"
        ground_truth[filename] = text
        
    print(f"✅ Successfully loaded {len(ground_truth)} reference sentences.")
    return ground_truth

txt_path = "/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/unseen_test_script.txt"
ground_truth_map = load_ground_truth_map(txt_path)
for k, v in list(ground_truth_map.items())[:3]:
    print(f"{k}: {v}")

# Error Type Analysis

In [ ]:
def analyze_asr_error_types(tts_results_dict):
    """
    Takes the output dictionary from validate_tts_with_asr and calculates
    the total Insertions (I), Deletions (D), and Substitutions (S).
    """
    file_details = tts_results_dict.get("file_details", [])
    
    total_sub = 0
    total_del = 0
    total_ins = 0
    total_words = 0
    
    for item in file_details:
        ref = item.get("ref", "")
        hyp = item.get("hyp", "")
        
        # Skip empty references to avoid division by zero
        if not ref:
            continue
            
        output = jiwer.process_words(ref, hyp)
        
        total_sub += output.substitutions
        total_del += output.deletions
        total_ins += output.insertions
        
        # The total number of words in the reference text is (Substitutions + Deletions + Hits)
        total_words += (output.substitutions + output.deletions + output.hits)

    if total_words == 0:
        return "No valid words found to analyze."

    sub_pct = (total_sub / total_words) * 100
    del_pct = (total_del / total_words) * 100
    ins_pct = (total_ins / total_words) * 100
    
    print("\n" + "="*40)
    print("📊 ASR ERROR TYPE BREAKDOWN (FOR REBUTTAL) 📊")
    print(f"Total Reference Words: {total_words}")
    print("-" * 40)
    print(f"Substitutions (S): {total_sub} errors ({sub_pct:.2f}%)")
    print(f"Deletions (D):     {total_del} errors ({del_pct:.2f}%)")
    print(f"Insertions (I):    {total_ins} errors ({ins_pct:.2f}%)")
    print("-" * 40)
    
    total_wer_check = sub_pct + del_pct + ins_pct
    print(f"Total WER:         {total_wer_check:.2f}%")
    print("="*40)
    
    return {
        "substitutions": total_sub,
        "deletions": total_del,
        "insertions": total_ins,
        "sub_pct": sub_pct,
        "del_pct": del_pct,
        "ins_pct": ins_pct
    }

In [ ]:
asr_apha_male = validate_tts_with_asr("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/aphavoice_male_wavs", ground_truth_map)
error_breakdown = analyze_asr_error_types(asr_apha_male)

In [ ]:
asr_apha_female = validate_tts_with_asr("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/aphavoice_female_wavs", ground_truth_map)
error_breakdown = analyze_asr_error_types(asr_apha_female)

# Running the Test

In [ ]:
asr_parler = validate_tts_with_asr("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/parler_wavs", ground_truth_map)

In [ ]:
asr_apha_male = validate_tts_with_asr("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/aphavoice_male_wavs", ground_truth_map)

In [ ]:
asr_apha_female = validate_tts_with_asr("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/aphavoice_female_wavs", ground_truth_map)

In [ ]:
asr_vits = validate_tts_with_asr("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/vits_wavs", ground_truth_map)

In [ ]:
asr_xtts = validate_tts_with_asr("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/xtts_wavs", ground_truth_map)

In [ ]:
asr_bark = validate_tts_with_asr("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/bark_wavs", ground_truth_map)

In [ ]:
asr_microsoft = validate_tts_with_asr("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/microsoft_wavs", ground_truth_map)

In [ ]:
asr_matcha_base = validate_tts_with_asr("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/matcha_base_male_wavs", ground_truth_map)

# Diagram Visualization

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
model_names = ["M (Base)", "XTTS", "VITS", "SpeechT5", "Bark", "Parler", "Apha-F*", "Apha-M*"]
models = [asr_matcha_base, asr_xtts, asr_vits, asr_microsoft, asr_bark, asr_parler, asr_apha_female, asr_apha_male]

data = []

# Loop through and grab the single true corpus-level aggregate per model
for name, model_data in zip(model_names, models):
    # Add single Corpus-level WER score
    data.append({
        "Model": name, 
        "Metric": "Word Error Rate (WER)", 
        "Error (%)": model_data["average_wer"] * 100
    })
    # Add single Corpus-level CER score
    data.append({
        "Model": name, 
        "Metric": "Character Error Rate (CER)", 
        "Error (%)": model_data["average_cer"] * 100
    })

df_asr = pd.DataFrame(data)

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=df_asr, 
    x="Model", 
    y="Error (%)", 
    hue="Metric", 
    palette=["#4C72B0", "#C44E52"],
    alpha=0.8
)

# Reference baseline thresholds
plt.axhline(y=24.3, color='blue', linestyle='--', linewidth=1.2, alpha=0.6)
plt.axhline(y=14.4, color='red', linestyle='--', linewidth=1.2, alpha=0.6)

plt.ylabel("Error Rate (%)", fontsize=14)
plt.xlabel("")
plt.ylim(0, 40)
plt.tick_params(axis='x', labelsize=16)
plt.tick_params(axis='y', labelsize=14)

for label in ax.get_xticklabels():
    if "Apha" in label.get_text():
        label.set_fontweight("bold")

plt.legend(title="", fontsize=14, loc="upper left")

plt.tight_layout()
plt.savefig("asr_intellygibility.png", bbox_inches="tight")
plt.show()

# Run Statistical Comparison

In [ ]:
import numpy as np
from scipy import stats

In [ ]:
import numpy as np
from scipy import stats

def run_statistical_comparison(model_1_data, model_2_data, model_1_name="AphaVoice", model_2_name="Matcha-Base"):
    """
    Runs a Mann-Whitney U test to compare the sentence-level WER and CER distributions of two models,
    while explicitly reporting the true corpus-level benchmark averages for your publication tables.
    """
    # 1. Extract the raw lists required for the Mann-Whitney U distribution test
    wer_1 = model_1_data["raw_wer_list"]
    cer_1 = model_1_data["raw_cer_list"]
    
    wer_2 = model_2_data["raw_wer_list"]
    cer_2 = model_2_data["raw_cer_list"]
    
    # 2. Extract the true corpus-level means (pooled micro-averages)
    corpus_wer_1 = model_1_data["average_wer"]
    corpus_wer_2 = model_2_data["average_wer"]
    
    corpus_cer_1 = model_1_data["average_cer"]
    corpus_cer_2 = model_2_data["average_cer"]
    
    # 3. Run the Mann-Whitney U tests on the sentence-level distributions
    # 'two-sided' tests if the distributions are significantly different in either direction
    u_stat_wer, p_value_wer = stats.mannwhitneyu(wer_1, wer_2, alternative="two-sided")
    u_stat_cer, p_value_cer = stats.mannwhitneyu(cer_1, cer_2, alternative="two-sided")
    
    # 4. Print the formatted results for your paper
    print("=" * 60)
    print(f"🔬 MANN-WHITNEY U TEST: {model_1_name} vs {model_2_name} 🔬")
    print("=" * 60)
    
    # --- WER Results ---
    print("\n[ Word Error Rate (WER) ]")
    print(f"🏆 {model_1_name} True Corpus Mean (Report This): {corpus_wer_1:.1%}")
    print(f"🏆 {model_2_name} True Corpus Mean (Report This): {corpus_wer_2:.1%}")
    print(f"   {model_1_name} Sentence-Level Macro Mean:       {np.mean(wer_1):.1%}")
    print(f"   {model_2_name} Sentence-Level Macro Mean:       {np.mean(wer_2):.1%}")
    print(f"📊 U-Statistic: {u_stat_wer}")
    print(f"📊 P-Value:     {p_value_wer:.5f}")
    
    if p_value_wer < 0.05:
        print("✅ Result: The difference in WER is STATISTICALLY SIGNIFICANT (p < 0.05).")
    else:
        print("❌ Result: The difference in WER is NOT statistically significant (p >= 0.05).")
        
    # --- CER Results ---
    print("\n[ Character Error Rate (CER) ]")
    print(f"🏆 {model_1_name} True Corpus Mean (Report This): {corpus_cer_1:.1%}")
    print(f"🏆 {model_2_name} True Corpus Mean (Report This): {corpus_cer_2:.1%}")
    print(f"   {model_1_name} Sentence-Level Macro Mean:       {np.mean(cer_1):.1%}")
    print(f"   {model_2_name} Sentence-Level Macro Mean:       {np.mean(cer_2):.1%}")
    print(f"📊 U-Statistic: {u_stat_cer}")
    print(f"📊 P-Value:     {p_value_cer:.5f}")
    
    if p_value_cer < 0.05:
        print("✅ Result: The difference in CER is STATISTICALLY SIGNIFICANT (p < 0.05).")
    else:
        print("❌ Result: The difference in CER is NOT statistically significant (p >= 0.05).")
    print("\n" + "=" * 60)

In [ ]:
run_statistical_comparison(
    model_1_data=asr_apha_male, 
    model_2_data=asr_matcha_base, 
    model_1_name="AphaVoice (Male)", 
    model_2_name="Matcha-TTS (Base Male)"
)